# Weaver SDK Interactive Walkthrough

This notebook demonstrates the core concepts of the Weaver SDK using a **Pig Latin** translation task,
explaining what happens behind the scenes at every step.

### Architecture Overview

```
┌──────────────────────┐        HTTP/REST         ┌────────────────────────┐
│   Your Code (SDK)    │  ◄─────────────────────►  │   Weaver Server        │
│                      │                           │                        │
│  ServiceClient       │  create_model ──────────► │  Provisioner (sched.)  │
│    └─ TrainingClient  │  forward_backward ─────► │    └─ Trainer (GPU)    │
│    └─ SamplingClient  │  optim_step ───────────► │    └─ Inference (GPU)  │
│                      │  save_state / load_state►│    └─ Storage           │
└──────────────────────┘                           └────────────────────────┘
```

**Key insight**: The SDK is a lightweight HTTP client. All heavy computation
(forward/backward passes, optimizer updates, weight storage) runs on **server-side GPUs**.
The SDK orchestrates the training loop locally.

### What This Notebook Covers

| # | Topic | What Happens on the Server |
|---|-------|---------------------------|
| 1 | Data Preparation | Tokenize text into `Datum` objects |
| 2 | LoRA Training | Load base model + LoRA adapters on GPU |
| 3 | Full Fine-Tuning | Load full model weights (all params trainable) |
| 4 | Sampling Client | Export weights to inference engine, generate text |
| 5 | save_state / load_state | Checkpoint management for training resumption |

---
## Install the Weaver SDK

This also installs dependencies: `torch`, `transformers`, `httpx`, etc.

In [ ]:
%pip install nex-weaver --upgrade

---
## 0. Environment Setup

In [ ]:
import os
from typing import Any, Dict, List, Tuple, Sequence

import torch

from weaver import ServiceClient, types

API_KEY = os.getenv("WEAVER_API_KEY")
BASE_MODEL = "Qwen/Qwen3-8B"

assert API_KEY, "Please set the WEAVER_API_KEY environment variable"

---
## 1. Data Preparation

Weaver operates on **token-level** data. Each training sample is a `Datum` containing:
- `model_input`: the input token sequence fed to the model
- `loss_fn_inputs`: a dict of tensors passed to the server-side loss function (e.g., target tokens, per-token weights)

For SFT we use standard **next-token prediction**:
- `input_tokens = tokens[:-1]` (drop last token)
- `target_tokens = tokens[1:]` (shift right by one)
- `weights`: 0.0 for prompt tokens (no loss), 1.0 for completion tokens

In [ ]:
EXAMPLES: List[Dict[str, str]] = [
    {"input": "banana split", "output": "anana-bay plit-say"},
    {"input": "quantum physics", "output": "uantum-qay ysics-phay"},
    {"input": "donut shop", "output": "onut-day op-shay"},
    {"input": "pickle jar", "output": "ickle-pay ar-jay"},
    {"input": "space exploration", "output": "ace-spay exploration-way"},
    {"input": "rubber duck", "output": "ubber-ray uck-day"},
    {"input": "coding wizard", "output": "oding-cay izard-way"},
]


def process_example(example: Dict[str, str], tokenizer) -> types.Datum:
    """Convert a text pair into a token-level Datum for Weaver."""
    prompt = f"English: {example['input']}\nPig Latin:"
    prompt_tokens = tokenizer.encode(prompt, add_special_tokens=True)
    completion_tokens = tokenizer.encode(f" {example['output']}\n\n", add_special_tokens=False)

    tokens = prompt_tokens + completion_tokens
    weights = [0.0] * len(prompt_tokens) + [1.0] * len(completion_tokens)

    input_tokens = tokens[:-1]
    target_tokens = tokens[1:]
    weights = weights[1:]

    return types.Datum(
        model_input=types.ModelInput.from_ints(input_tokens),
        loss_fn_inputs={
            "target_tokens": torch.tensor(target_tokens, dtype=torch.int64),
            "weights": torch.tensor(weights, dtype=torch.float32),
        },
    )


def visualize_datum(datum: types.Datum, tokenizer) -> None:
    """Show token-level input / target / weight alignment."""
    print(f"{'Input':<20} {'Target':<20} {'Weight':<10}")
    print("-" * 50)
    for inp, tgt, wgt in zip(
        datum.model_input.to_ints(),
        datum.loss_fn_inputs["target_tokens"].tolist(),
        datum.loss_fn_inputs["weights"].tolist(),
    ):
        marker = "  <-- loss computed here" if wgt > 0 else ""
        print(f"{repr(tokenizer.decode([inp])):<20} {repr(tokenizer.decode([tgt])):<20} {wgt:<10}{marker}")

In [ ]:
def extract_logprobs(output: Dict[str, Any]) -> torch.Tensor:
    """Extract logprobs from forward/backward result."""
    value = output.get("logprobs") or output.get("Logprobs")
    if isinstance(value, dict):
        value = value.get("data")
    if value is None:
        raise ValueError("logprobs missing from forward/backward result")
    return torch.as_tensor(value, dtype=torch.float32)


def compute_loss(fwdbwd_result: dict, processed_examples: list) -> float:
    """Compute weighted cross-entropy loss locally (for monitoring)."""
    outputs = fwdbwd_result.get("result", {}).get("loss_fn_outputs") or []
    logprobs = torch.cat([extract_logprobs(o) for o in outputs], dim=0)
    weights = torch.cat(
        [ex.loss_fn_inputs["weights"] for ex in processed_examples], dim=0
    )
    return float(-torch.dot(logprobs, weights) / weights.sum())

---
## 2. LoRA Training

LoRA (Low-Rank Adaptation) freezes the base model weights and injects small trainable matrices.

### What happens on the server when `create_model()` is called (default = LoRA):
1. SDK sends `POST /api/v1/sessions/{session_id}/models` with `lora_config`
2. Server allocates a **Trainer GPU instance**
3. Loads base model weights and initializes LoRA adapters (only these params are trainable)
4. Returns a `TrainingClient` handle

### LoRA Configuration:
- `rank`: dimension of the low-rank matrices (default=32)
- `train_attn`: apply LoRA to attention layers (default=True)
- `train_mlp`: apply LoRA to MLP/MoE layers (default=True)
- `train_unembed`: apply LoRA to the output projection (default=True)
- `seed`: random seed for LoRA weight initialization

In [ ]:
service_client = ServiceClient(api_key=API_KEY)
service_client.connect()
print("Connected to Weaver service.")

In [ ]:
# create_model() defaults to LoRA training mode (rank=32)
# Customize with: lora_config=types.LoraConfig(rank=16, train_attn=True, train_mlp=False)

lora_client = service_client.create_model(
    base_model=BASE_MODEL,
    lora_config=types.LoraConfig(rank=32),
)
print(f"Model ID: {lora_client.model_id}")
print(f"Base model: {lora_client.base_model}")

In [ ]:
tokenizer = lora_client.get_tokenizer()

processed_examples = [process_example(ex, tokenizer) for ex in EXAMPLES]
print(f"Prepared {len(processed_examples)} training samples\n")

visualize_datum(processed_examples[0], tokenizer)

### forward_backward + optim_step = One Mini-Batch

In traditional training, a mini-batch step consists of: forward → loss → backward → optimizer.step().

Weaver maps directly to this:

```
┌─────────────────────────────────────────────────────────────────┐
│  forward_backward(data, "cross_entropy")                       │
│                                                                 │
│    SDK sends data (N Datums) ──►  Server (Trainer GPU)          │
│                                                                 │
│    Server internally:                                           │
│      1. Splits N Datums into micro-batches                      │
│         (bounded by GPU memory; each micro-batch runs fwd+bwd)  │
│      2. Accumulates gradients across all micro-batches          │
│      3. Returns per-Datum logprobs to SDK (for local monitoring)│
│                                                                 │
│    ⚠️ forward_backward does NOT split mini-batches!             │
│    The data you pass IS the full mini-batch.                    │
│    The server only splits into micro-batches.                   │
├─────────────────────────────────────────────────────────────────┤
│  optim_step(adam_params)                                        │
│                                                                 │
│    Server internally:                                           │
│      1. Performs one Adam update with accumulated gradients      │
│      2. Zeros gradients for the next mini-batch                 │
│                                                                 │
│    ⚠️ One forward_backward + one optim_step = one weight update │
└─────────────────────────────────────────────────────────────────┘
```

**Mapping to traditional training:**

| Traditional Concept | Weaver Equivalent | Who Is Responsible |
|---------------------|-------------------|--------------------|
| Mini-batch (N samples) | `data` in `forward_backward(data)` | User / NexRL |
| Micro-batch (memory split) | Automatic server-side split | Weaver Server |
| Gradient accumulation across micro-batches | Automatic | Weaver Server |
| Multiple mini-batches → larger effective batch | Multiple `forward_backward` calls before `optim_step` | User / NexRL |
| optimizer.step() | `optim_step(adam_params)` | Weaver Server |

**Advanced: Gradient accumulation across multiple mini-batches**

To accumulate gradients over multiple mini-batches before updating (equivalent to a larger effective batch size),
call `forward_backward` multiple times then call `optim_step` once:

```python
# effective_batch = mini_batch_1 + mini_batch_2 + mini_batch_3
for mini_batch in [mini_batch_1, mini_batch_2, mini_batch_3]:
    training_client.forward_backward(mini_batch, "cross_entropy", wait=True)

# Gradients accumulated across all 3 forward_backward calls
training_client.optim_step(adam, wait=True)  # single weight update
```

In [ ]:
# Each iteration: forward_backward + optim_step = one mini-batch step
# processed_examples (7 samples) is the full mini-batch
# The server automatically splits into micro-batches internally

adam = types.AdamParams(learning_rate=1e-4)

print("=== LoRA Training (each step = 1 mini-batch) ===")
for step in range(3):
    fwdbwd_result = lora_client.forward_backward(
        processed_examples,
        "cross_entropy",
        wait=True,
    )
    lora_client.optim_step(adam, wait=True)

    loss = compute_loss(fwdbwd_result, processed_examples)
    print(f"Step {step}: loss/token = {loss:.4f}")

print("\nLoRA training complete.")

---
## 3. Full Fine-Tuning

Full fine-tuning makes **all** model parameters trainable (no LoRA adapters).

### Differences from LoRA:
- `training_mode="full_ft"` → server does not create LoRA adapters
- All transformer weights are directly updated by the optimizer
- Typically requires a **smaller learning rate** (e.g., 1e-5) to avoid catastrophic forgetting
- Requires more GPU memory on the server

In [ ]:
fullft_client = service_client.create_model(
    base_model=BASE_MODEL,
    training_mode="full_ft",
)
print(f"Full FT Model ID: {fullft_client.model_id}")

In [ ]:
tokenizer_ft = fullft_client.get_tokenizer()
processed_examples_ft = [process_example(ex, tokenizer_ft) for ex in EXAMPLES]

adam_ft = types.AdamParams(learning_rate=1e-5)

print("=== Full Fine-Tuning ===")
for step in range(3):
    fwdbwd_result = fullft_client.forward_backward(
        processed_examples_ft, "cross_entropy", wait=True
    )
    fullft_client.optim_step(adam_ft, wait=True)

    loss = compute_loss(fwdbwd_result, processed_examples_ft)
    print(f"Step {step}: loss/token = {loss:.4f}")

print("\nFull fine-tuning complete.")

---
## 4. Sampling Client

After training, you need to **export** the trained weights to an inference engine to generate text.

### What happens on the server:
1. `save_weights_and_get_sampling_client()` → sends an export request to the server
2. Server converts trained weights (LoRA-merged or full) into an inference-optimized format
3. Creates an inference GPU session
4. Returns a `SamplingClient` you can call `sample()` on

### Two-step alternative:
```python
model_path = training_client.save_weights_for_sampler(name="my-model")
sampling_client = service_client.create_sampling_client(base_model=..., model_path=model_path)
```

In [ ]:
sampling_client = lora_client.save_weights_and_get_sampling_client(
    name="pig-latin-lora-model"
)
print(f"Sampling client ready. Model path: {sampling_client.model_path}")

In [ ]:
test_inputs = ["coffee break", "hello world", "machine learning"]

for text in test_inputs:
    prompt_str = f"English: {text}\nPig Latin:"
    prompt_tokens = tokenizer.encode(prompt_str, add_special_tokens=True)
    prompt = types.ModelInput.from_ints(prompt_tokens)

    params = types.SamplingParams(
        max_tokens=20,
        temperature=0.0,
        stop=["\n"],
    )
    result = sampling_client.sample(
        prompt=prompt,
        sampling_params=params,
        num_samples=1,
    )
    decoded = tokenizer.decode(result["sequences"][0].get("tokens", []))
    print(f"{text:>20}  →  {decoded.strip()}")

---
## 5. Checkpoint Management: save_state / load_state

Weaver supports saving and loading model checkpoints for training resumption.

### Checkpoint Types:
- `"weight"` (default): saves model weights only
- `"weight_and_optimizer"`: saves weights + Adam momentum/variance state

### API Methods:
| Method | Purpose |
|--------|---------|
| `save_state()` | Server writes checkpoint to storage, returns a `Checkpoint` object |
| `load_state(path)` | Restores weights only (optimizer state is reset) |
| `load_state_with_optimizer(path)` | Restores weights + optimizer state (true resume) |
| `list_checkpoints()` | Lists all checkpoints for this model |

### Server Behavior:
```
save_state(name="step-6"):
  POST /api/v1/models/{model_id}/checkpoints
  → Trainer writes weights to: weaver://{model_id}/checkpoints/step-6
  → Returns Checkpoint(id=..., path="weaver://...", name="step-6")

load_state(checkpoint):
  POST /api/v1/models/{model_id}/load
  Body: {path: "weaver://...", include_optimizer: false}
  → Trainer loads weights from storage, replacing current weights
```

In [ ]:
ckpt_weight = lora_client.save_state(name="lora-step-9", checkpoint_type="weight")
print(f"Saved checkpoint:")
print(f"  id:   {ckpt_weight.id}")
print(f"  path: {ckpt_weight.path}")
print(f"  name: {ckpt_weight.name}")
print(f"  type: {ckpt_weight.checkpoint_type}")

In [ ]:
ckpt_full = lora_client.save_state(
    name="lora-step-9-with-optim",
    checkpoint_type="weight_and_optimizer",
)
print(f"Saved full checkpoint: {ckpt_full.path}")

In [ ]:
checkpoints = lora_client.list_checkpoints()
print(f"{len(checkpoints)} checkpoint(s):")
for ckpt in checkpoints:
    print(f"  [{ckpt.checkpoint_type:>22}] {ckpt.name or '(unnamed)':>30}  →  {ckpt.path}")

In [ ]:
# load_state: restore weights only (optimizer state is reset)
# Use case: roll back to an earlier checkpoint, then continue training with fresh optimizer
lora_client.load_state(ckpt_weight)
print(f"Loaded weights from: {ckpt_weight.path}")
print("Optimizer state was reset.")

In [ ]:
# load_state_with_optimizer: true resume (weights + Adam momentum/variance)
# Use case: resume training exactly where you left off
lora_client.load_state_with_optimizer(ckpt_full)
print(f"Loaded weights + optimizer from: {ckpt_full.path}")
print("Training can resume with preserved optimizer state.")

In [ ]:
print("=== Resumed Training ===")
adam_resumed = types.AdamParams(learning_rate=1e-4)
for step in range(3):
    fwdbwd_result = lora_client.forward_backward(
        processed_examples, "cross_entropy", wait=True
    )
    lora_client.optim_step(adam_resumed, wait=True)

    loss = compute_loss(fwdbwd_result, processed_examples)
    print(f"Resumed step {step}: loss/token = {loss:.4f}")

print("\nResumed training complete.")

---
## 6. Cleanup

`terminate()` releases the GPU instances (trainer + inference) provisioned for each model.
The `ServiceClient` context manager also sends a session close on exit.

In [ ]:
lora_client.terminate()
fullft_client.terminate()
service_client.close()
print("All resources released.")

---
## Quick Reference

| SDK Call | Server-Side Effect |
|----------|---------|
| `ServiceClient(api_key=...)` | Establishes authenticated session |
| `create_model(base_model=..., training_mode=None)` | Provisions GPU, loads model + LoRA adapters |
| `create_model(base_model=..., training_mode="full_ft")` | Provisions GPU, loads model (all params trainable) |
| `forward_backward(data, "cross_entropy")` | Forward pass + loss + backward pass on GPU |
| `optim_step(AdamParams(...))` | Adam update on accumulated gradients |
| `save_weights_and_get_sampling_client()` | Export weights → create inference session |
| `sampling_client.sample(prompt, params)` | Generate text using trained model |
| `save_state(name=..., checkpoint_type=...)` | Write checkpoint to server storage |
| `load_state(checkpoint)` | Restore weights (optimizer reset) |
| `load_state_with_optimizer(checkpoint)` | Restore weights + optimizer (true resume) |
| `terminate()` | Release GPU resources |